In [1]:
!pip install PyPDF2 python-docx nltk scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.2 MB/s eta 0:00:00


In [3]:
import os
import re
import nltk
import pandas as pd

from google.colab import files

from PyPDF2 import PdfReader
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
uploaded = files.upload()

Saving resume1_data_science.pdf to resume1_data_science.pdf


In [5]:
def extract_text_from_pdf(file_path):
    text = ""

    try:
        reader = PdfReader(file_path)

        for page in reader.pages:
            extracted = page.extract_text()

            if extracted:
                text += extracted

    except Exception as e:
        print(f"Error reading PDF {file_path}: {e}")

    return text


def extract_text_from_docx(file_path):
    text = ""

    try:
        doc = Document(file_path)

        for para in doc.paragraphs:
            text += para.text + "\n"

    except Exception as e:
        print(f"Error reading DOCX {file_path}: {e}")

    return text


def extract_text_from_txt(file_path):
    text = ""

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()

    except Exception as e:
        print(f"Error reading TXT {file_path}: {e}")

    return text

In [6]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+', ' ', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove stopwords
    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [7]:
resume_data = []

for filename in uploaded.keys():

    text = ""

    if filename.endswith(".pdf"):
        text = extract_text_from_pdf(filename)

    elif filename.endswith(".docx"):
        text = extract_text_from_docx(filename)

    elif filename.endswith(".txt"):
        text = extract_text_from_txt(filename)

    else:
        print(f"Unsupported file format: {filename}")
        continue

    cleaned_text = clean_text(text)

    resume_data.append({
        "Resume_Name": filename,
        "Resume_Text": cleaned_text
    })

print("All resumes processed successfully!")

All resumes processed successfully!


The previous error occurred because the uploaded file was a `.zip` archive, and the processing logic only handled `.pdf`, `.docx`, and `.txt` files directly. To fix this, I will add a function to extract files from the zip archive and then iterate through the extracted files for processing.

In [8]:
import zipfile
import tempfile

In [9]:
def extract_from_zip(zip_file_path):
    extracted_file_paths = []
    temp_dir = tempfile.mkdtemp()

    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
            for root, _, files in os.walk(temp_dir):
                for file in files:
                    extracted_file_paths.append(os.path.join(root, file))
    except Exception as e:
        print(f"Error extracting zip file {zip_file_path}: {e}")
    return extracted_file_paths, temp_dir

In [12]:
job_description = """
We are looking for a Python Developer with experience in:
Python,
Machine Learning,
Data Analysis,
SQL,
Deep Learning,
NLP,
Flask,
APIs,
Git,
Data Visualization
"""
cleaned_jd = clean_text(job_description)

resume_data = []
all_files_to_process = []

for filename, content in uploaded.items():
    if filename.endswith('.zip'):
        # Save the uploaded BytesIO content to a temporary zip file
        with open(filename, 'wb') as f:
            f.write(content)
        extracted_files, temp_dir = extract_from_zip(filename)
        all_files_to_process.extend([(f, temp_dir) for f in extracted_files])

    else:
        all_files_to_process.append((filename, None))


for file_path, temp_dir in all_files_to_process:
    text = ""
    original_filename = os.path.basename(file_path)

    if file_path.endswith(".pdf"):
        text = extract_text_from_pdf(file_path)
    elif file_path.endswith(".docx"):
        text = extract_text_from_docx(file_path)
    elif file_path.endswith(".txt"):
        text = extract_text_from_txt(file_path)
    else:
        print(f"Unsupported file format: {original_filename}")
        continue

    cleaned_text = clean_text(text)

    resume_data.append({
        "Resume_Name": original_filename,
        "Resume_Text": cleaned_text
    })

    if temp_dir and os.path.exists(temp_dir):
        import shutil
        shutil.rmtree(temp_dir)


if resume_data:
    print("All resumes processed successfully!")
else:
    print("No resumes were processed. Please check file formats.")


resume_texts = [resume["Resume_Text"] for resume in resume_data]

if not resume_texts:
    print("No resume texts available for TF-IDF vectorization. Please upload valid resume files.")
else:
    documents = [cleaned_jd] + resume_texts

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity_scores = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:]
    )
    scores = similarity_scores[0]
    print("TF-IDF vectorization and cosine similarity calculation complete.")

All resumes processed successfully!
TF-IDF vectorization and cosine similarity calculation complete.


In [13]:
job_description = """
We are looking for a Python Developer with experience in:
Python,
Machine Learning,
Data Analysis,
SQL,
Deep Learning,
NLP,
Flask,
APIs,
Git,
Data Visualization
"""

In [14]:
cleaned_jd = clean_text(job_description)

In [15]:
resume_texts = [resume["Resume_Text"] for resume in resume_data]

if not resume_texts:
    print("No resume texts available for TF-IDF vectorization. Please upload valid resume files.")
else:
    documents = [cleaned_jd] + resume_texts

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity_scores = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:]
    )

    scores = similarity_scores[0]
    print("TF-IDF vectorization and cosine similarity calculation complete.")

TF-IDF vectorization and cosine similarity calculation complete.


In [16]:
results = []

# Ensure scores are available before proceeding
if 'scores' in locals() and scores.size > 0:
    for i, resume in enumerate(resume_data):

        score = round(scores[i] * 100, 2)

        results.append({
            "Resume_Name": resume["Resume_Name"],
            "Match_Score": score
        })

results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(
        by="Match_Score",
        ascending=False
    )

    results_df.reset_index(drop=True, inplace=True)

    print("Match scores calculated and sorted successfully!")
    # Display the DataFrame
    print(results_df.to_markdown(index=False))
else:
    print("No resumes were processed or no match scores could be calculated. Please check your uploaded files.")

Match scores calculated and sorted successfully!
| Resume_Name              |   Match_Score |
|:-------------------------|--------------:|
| resume1_data_science.pdf |         35.68 |


In [17]:
if not results_df.empty:
    top_resume = results_df.iloc[0]

    print("Best Matching Resume")
    print("----------------------")
    print(f"Resume Name : {top_resume['Resume_Name']}")
    print(f"Match Score : {top_resume['Match_Score']}%")
else:
    print("No matching resumes found or no resumes were processed to determine the best match.")

Best Matching Resume
----------------------
Resume Name : resume1_data_science.pdf
Match Score : 35.68%


In [18]:
results_df.to_csv("resume_screening_results.csv", index=False)

files.download("resume_screening_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>